# Analisi delle variabili consigliate - Gruppo 3

Slide 10, piste sulle feature: **distanza venditore-cliente, peso e volume, tempo di approvazione del pagamento, categoria, carico del venditore nel periodo.**

Ogni feature e' costruita in `src/features.py` (a grana-ordine) e qui ne guardiamo copertura, segno e forza della relazione col target `delivery_days`.

> Regole seguite: distanza come **media e max** sugli articoli (il target e' l'arrivo dell'ultimo articolo); carico venditore su finestra **strettamente causale** (solo ordini precedenti); per ogni feature si riporta la **copertura (% non-nulli)** perche' la correlazione su un sottoinsieme distorto inganna.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
from src import features
pd.set_option('display.width', 120)

## Tabella feature
`build_feature_table()` = tabella ordine (target + baseline) + le 5 feature consigliate.

In [2]:
t = features.build_feature_table()
print(f'{t.shape[0]:,} ordini x {t.shape[1]} colonne')
t.filter(regex='distance|weight|volume|approval|seller_load|main_category|delivery_days').head()

96,470 ordini x 31 colonne


,delivery_days,distance_km_mean,distance_km_max,total_weight_g,total_volume_cm3,approval_hours,main_category,seller_load_mean,seller_load_max
0,8.436574,18.681711,18.681711,500.0,1976.0,0.178333,housewares,7.0,7
1,13.782037,861.035367,861.035367,400.0,4693.0,30.713889,perfumery,24.0,24
2,9.394213,514.547140,514.547140,420.0,9576.0,0.276111,auto,83.0,83
3,13.208750,1821.802656,1821.802656,450.0,6000.0,0.298056,pet_shop,7.0,7
4,2.873877,29.593095,29.593095,250.0,11475.0,1.030556,stationery,8.0,8


## 1. Copertura e correlazione col target
Spearman (monotona, robusta): quanto ogni feature numerica si muove col target.

In [3]:
y = 'delivery_days'
num = ['distance_km_mean','distance_km_max','total_weight_g','total_volume_cm3',
       'approval_hours','seller_load_mean','seller_load_max']
rows = []
for col in num:
    cov = t[col].notna().mean()*100
    sp = t[[col, y]].dropna().corr(method='spearman').iloc[0, 1]
    rows.append((col, f'{cov:.1f}%', round(sp, 3), round(t[col].median(), 1)))
pd.DataFrame(rows, columns=['feature', 'copertura', 'spearman_r', 'mediana'])

,feature,copertura,spearman_r,mediana
0,distance_km_mean,99.5%,0.543,433.8
1,distance_km_max,99.5%,0.541,435.5
2,total_weight_g,100.0%,0.087,750.0
3,total_volume_cm3,100.0%,0.069,7250.0
4,approval_hours,100.0%,0.088,0.3
5,seller_load_mean,100.0%,0.050,13.0
6,seller_load_max,100.0%,0.048,13.0


## 2. Distanza - controllo qualita'
Il Brasile si estende per ~4300 km: distanze oltre indicano coordinate geolocation sporche (problema noto del dataset), non un errore di calcolo.

In [4]:
print(t['distance_km_mean'].describe()[['min','50%','max']])
bad = (t['distance_km_max'] > 4300).sum()
print(f'\nordini con distanza > 4300 km: {bad}  ({bad/t["distance_km_max"].notna().sum()*100:.2f}%)')

min       0.000000
50%     433.833161
max    8677.859564
Name: distance_km_mean, dtype: float64

ordini con distanza > 4300 km: 4  (0.00%)


## 3. Categoria vs consegna
La categoria non ha correlazione: guardiamo la consegna media per categoria.

In [5]:
g = t.groupby('main_category')['delivery_days'].agg(['mean','count'])
g = g[g['count'] >= 300].sort_values('mean')
print('Piu veloci:'); print(g.head(5).round(1))
print('\nPiu lente:'); print(g.tail(5).round(1))

Piu veloci:
                                 mean  count
main_category                               
food                              9.8    436
construction_tools_construction  10.9    731
luggage_accessories              10.9   1011
small_appliances                 11.0    604
housewares                       11.1   5671

Piu lente:
                       mean  count
main_category                     
garden_tools           13.7   3411
home_confort           13.8    346
consoles_games         13.8   1016
furniture_living_room  14.1    403
office_furniture       20.7   1244


## 4. Carico venditore - controllo causalita'
Il primo ordine di un venditore deve avere carico 0, poi cresce con gli ordini precedenti nella finestra di 30 giorni (nessun ordine futuro).

In [6]:
from src import data_loader
raw = data_loader.load_all_raw()
sv = raw['order_items'][['order_id','seller_id']].drop_duplicates().merge(
    raw['orders'][['order_id','order_purchase_timestamp']], on='order_id')
top = sv['seller_id'].value_counts().index[0]
g = sv[sv.seller_id == top].merge(t[['order_id','seller_load_max']], on='order_id')
g.sort_values('order_purchase_timestamp')[['order_purchase_timestamp','seller_load_max']].head(6)

,order_purchase_timestamp,seller_load_max
796,2017-02-17 07:39:19,0
1480,2017-02-23 10:36:35,1
1228,2017-02-24 20:28:53,2
31,2017-02-27 09:27:13,3
120,2017-02-28 20:57:21,4
1413,2017-03-01 20:28:48,5


## Sintesi
- **Distanza**: driver piu' forte del target (Spearman ~0.54). Media e max quasi identici -> la distanza al venditore piu' lontano coincide spesso con quella media.
- **Categoria**: forte spread (consegna media ~10 vs ~21 giorni tra categorie).
- **Peso/volume, tempo di approvazione, carico venditore**: correlazioni deboli (<0.1) singolarmente; possono comunque contare in un modello non lineare.

**TODO (modellazione):** encoding della categoria, gestione dei nulli di distanza, confronto col baseline `estimated_days`, split temporale prima di consolidare il carico venditore come feature. Motivare le scelte nel README.